In [1]:
#!pip install transformers datasets torch scikit-learn pandas

In [ ]:
import pandas as pd
import numpy as np
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.model_selection import train_test_split
from pathlib import Path

from transformers import EarlyStoppingCallback

In [3]:
possible_paths = [
    Path('../dataset/dataset_clean_final.csv'),
    Path('dataset_clean_final.csv'),
    Path('/content/dataset_clean_final.csv')
]

for path in possible_paths:
    if path.exists():
        df = pd.read_csv(path)
        print(f"Dataset berhasil dibaca dari: {path}")
        break
else:
    raise FileNotFoundError('File dataset_clean_final.csv tidak ditemukan. Pastikan file berada di folder dataset atau satu folder dengan notebook.')

Dataset berhasil dibaca dari: ..\dataset\dataset_clean_final.csv


In [4]:
# Load dataset
df = df.dropna(subset=['text', 'label'])
df.head()

,text,label
0,makin yakin habis baca review lain tentang vic...,1
1,paling suka model h2 smiling_face_with_heart e...,0
2,mobilnya sudah hancur pleading_face,0
3,manut88benar2 bikin aku jadi sultan,1
4,semoga lekas recover mobilnya mas dipo,0


In [5]:
# Membagi dataset menjadi 80% Training dan 20% Validation
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df['text'].tolist(),
    df['label'].tolist(),
    test_size=0.2,
    random_state=42,
    stratify=df['label'] # Menjaga rasio label tetap seimbang
)

# Konversi format list biasa menjadi format 'Dataset' bawaan Hugging Face
train_dataset = Dataset.from_dict({'text': train_texts, 'label': train_labels})
val_dataset = Dataset.from_dict({'text': val_texts, 'label': val_labels})

print(f"Jumlah data latih: {len(train_dataset)}")
print(f"Jumlah data validasi: {len(val_dataset)}")

Jumlah data latih: 56303
Jumlah data validasi: 14076


In [6]:
# Menggunakan pre-trained model IndoBERT
model_name = "indobenchmark/indobert-base-p1"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    # max_length=128 cukup ideal untuk komentar YouTube
    # padding & truncation memastikan semua input memiliki panjang seragam
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

# Menerapkan tokenisasi ke seluruh data latih dan validasi secara massal (batched)
tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_val = val_dataset.map(tokenize_function, batched=True)

print("Proses tokenisasi selesai!")

Map:   0%|          | 0/56303 [00:00<?, ? examples/s]

Map:   0%|          | 0/14076 [00:00<?, ? examples/s]

Proses tokenisasi selesai!


In [7]:
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)

    # Menghitung metrik klasifikasi biner
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)

    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

In [ ]:
# Memuat model IndoBERT dengan spesifikasi 2 label kelas klasifikasi
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# Konfigurasi argumen pelatihan
training_args = TrainingArguments(
    output_dir="./indobert_judol_model",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
)

# Inisialisasi API Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    processing_class=tokenizer,       # FIX: Mengubah 'tokenizer' menjadi 'processing_class'
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `5`.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: indobenchmark/indobert-base-p1
Key               | Status  | 
------------------+---------+-
classifier.bias   | MISSING | 
classifier.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [9]:
# Memulai proses training
print("Memulai proses fine-tuning IndoBERT...")
trainer.train()

Memulai proses fine-tuning IndoBERT...


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.028920,0.021115,0.996377,0.984303,0.990706,0.977982
2,0.016786,0.021668,0.996590,0.985285,0.987707,0.982875
3,0.004380,0.027173,0.996448,0.984653,0.988293,0.981040


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=10557, training_loss=0.020673893902694637, metrics={'train_runtime': 5390.3903, 'train_samples_per_second': 31.335, 'train_steps_per_second': 1.958, 'total_flos': 1.111045631245056e+16, 'train_loss': 0.020673893902694637, 'epoch': 3.0})

In [11]:
# Menentukan nama folder penyimpanan
model_path = "../model/indobert_judol_model"

# Menyimpan model dan tokenizer
trainer.save_model(model_path)
tokenizer.save_pretrained(model_path)

print(f"Model berhasil disimpan di folder: {model_path}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model berhasil disimpan di folder: ../model/indobert_judol_model


In [15]:
from transformers import pipeline

# Load model yang baru saja disimpan
classifier = pipeline("text-classification", model=model_path, tokenizer=model_path)

# Contoh teks pengujian
test_comments = [
    "Wah videonya sangat edukatif, terima kasih bang!",
    "Bongkar rahasia wd terus bosku, cek link di bio sekarang depo 10k jadi 100k",
    "Gacor banget bang mainnya, tutor dong"
]

# Jalankan prediksi
predictions = classifier(test_comments)

for text, pred in zip(test_comments, predictions):
    label = "JUDOL" if pred['label'] == 'LABEL_1' else "BUKAN JUDOL"
    print(f"Komentar: {text}")
    print(f"Prediksi: {label} (Confidence: {pred['score']:.4f})\n")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Komentar: Wah videonya sangat edukatif, terima kasih bang!
Prediksi: BUKAN JUDOL (Confidence: 0.9997)

Komentar: Bongkar rahasia wd terus bosku, cek link di bio sekarang depo 10k jadi 100k
Prediksi: BUKAN JUDOL (Confidence: 0.8262)

Komentar: Gacor banget bang mainnya, tutor dong
Prediksi: BUKAN JUDOL (Confidence: 0.9982)

